Fifth42: longer matched CPU batching test

Compare 1-, 4- and 8-window preparation batches at map chunk size 8, with eight workers, 12 replay passes per trial and five randomized matched repetitions. Use your current CUDA GPU. Run top to bottom; setup/testing is bounded to fifteen minutes and the benchmark to eight minutes. Every original exact check and durable receipt is retained. Zero new search coverage.


In [ ]:
import os, sys, time, subprocess, platform
from pathlib import Path
SESSION_DEADLINE = time.monotonic() + 900

def session_budget(maximum):
    left = SESSION_DEADLINE - time.monotonic()
    if left <= 0:
        raise TimeoutError("Fifteen-minute setup/test session limit reached")
    return min(maximum, left)

print("Notebook Python:", platform.python_version())
print("CPU affinity:", len(os.sched_getaffinity(0)))
subprocess.run(["nvidia-smi"], check=True, timeout=session_budget(10))
subprocess.run(["nvcc", "--version"], check=True, timeout=session_budget(10))
# If Colab shows a compute-unit rate, enter its numeric value here; otherwise leave None.
UNITS_PER_HOUR = None

if len(os.sched_getaffinity(0)) < 9:
    raise RuntimeError("This test requires eight CPU workers plus a coordinator; record the allocated CPU count and stop")


Download and verify the immutable original inputs and planning sources, then the longer batching test. No campus paths or manual uploads are needed.


In [ ]:
import base64, concurrent.futures, hashlib, io, json, tarfile, tempfile, urllib.request
ASSET_BASE = 'https://raw.githubusercontent.com/paulnord/fifth42/4668a27951f2b9d7d533896d1aa32684808341d4/colab/20261112'
EXPECTED_MANIFEST_SHA256 = '57d6965885da6c397f5cc1265fe71d15e7d1bc9c95434a3e166343f14f03bcae'
with urllib.request.urlopen(ASSET_BASE + "/transport/manifest.json", timeout=30) as response:
    manifest_raw = response.read()
assert hashlib.sha256(manifest_raw).hexdigest() == EXPECTED_MANIFEST_SHA256, "Manifest checksum mismatch"
manifest = json.loads(manifest_raw)
def fetch_part(entry):
    assert entry["name"].startswith("part-") and "/" not in entry["name"]
    with urllib.request.urlopen(ASSET_BASE + "/transport/" + entry["name"], timeout=30) as response:
        raw = response.read()
    assert len(raw) == entry["bytes"] and hashlib.sha256(raw).hexdigest() == entry["sha256"], "Chunk checksum mismatch"
    return raw
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    encoded_parts = list(pool.map(fetch_part, manifest["chunks"]))
archive_data = base64.b64decode(b"".join(encoded_parts), validate=True)
assert len(archive_data) == manifest["archive_bytes"]
assert hashlib.sha256(archive_data).hexdigest() == manifest["archive_sha256"], "Archive checksum mismatch"
FIFTH42_DIR = Path(tempfile.mkdtemp(prefix="fifth42-sustained-batching-", dir="/content"))
seen = set()
with tarfile.open(fileobj=io.BytesIO(archive_data), mode="r:gz") as archive:
    for member in archive.getmembers():
        rel = Path(member.name)
        assert member.isfile() and not rel.is_absolute() and ".." not in rel.parts and member.name not in seen
        seen.add(member.name)
        target = FIFTH42_DIR / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.extractfile(member).read())
for line in (FIFTH42_DIR / "SHA256SUMS").read_text().splitlines():
    digest, name = line.split("  ", 1)
    assert hashlib.sha256((FIFTH42_DIR / name).read_bytes()).hexdigest() == digest, name
print("Verified GitHub source/data:", FIFTH42_DIR)
print("Asset commit:", ASSET_BASE)


In [ ]:
PIPELINE_SOURCE_BASE = "https://raw.githubusercontent.com/paulnord/fifth42/99a6821796bc444e7541337edb0d1bf0815c8ad2/colab/20261112/pipeline"
EXPECTED_PIPELINE_MANIFEST = "20ea3d091882c1f3b468ee12ec657174dbea8756d95e804762580b8499e9f4d2"
with urllib.request.urlopen(PIPELINE_SOURCE_BASE + "/MANIFEST.json", timeout=session_budget(30)) as response:
    pipeline_manifest_raw = response.read()
assert hashlib.sha256(pipeline_manifest_raw).hexdigest() == EXPECTED_PIPELINE_MANIFEST
pipeline_manifest = json.loads(pipeline_manifest_raw)
pipeline_dir = FIFTH42_DIR / "pipeline"
pipeline_dir.mkdir()

def fetch_pipeline_file(entry):
    rel = Path(entry["path"])
    assert not rel.is_absolute() and ".." not in rel.parts
    with urllib.request.urlopen(PIPELINE_SOURCE_BASE + "/" + entry["path"], timeout=session_budget(30)) as response:
        data = response.read()
    assert len(data) == entry["bytes"] and hashlib.sha256(data).hexdigest() == entry["sha256"], entry["path"]
    target = pipeline_dir / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(data)

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(fetch_pipeline_file, pipeline_manifest["files"]))
(pipeline_dir / "MANIFEST.json").write_bytes(pipeline_manifest_raw)
print("Verified planner files:", len(pipeline_manifest["files"]))
print("Temporary runtime:", FIFTH42_DIR)


In [ ]:
BATCHING_SOURCE_BASE = "https://raw.githubusercontent.com/paulnord/fifth42/849d00a41d1520c7e1fcfdbc3e06d40e92fdcb70/colab/20261112/batching_sustained"
EXPECTED_BATCHING_MANIFEST = "5834066131a5b68bea83c7639510f2d3bf084faad7565ae434c5183ad7d2bdc2"
with urllib.request.urlopen(BATCHING_SOURCE_BASE + "/MANIFEST.json", timeout=session_budget(30)) as response:
    batching_manifest_raw = response.read()
assert hashlib.sha256(batching_manifest_raw).hexdigest() == EXPECTED_BATCHING_MANIFEST
batching_manifest = json.loads(batching_manifest_raw)
batching_dir = FIFTH42_DIR / "batching_sustained"
batching_dir.mkdir()

def fetch_batching_file(entry):
    rel = Path(entry["path"])
    assert not rel.is_absolute() and ".." not in rel.parts
    with urllib.request.urlopen(BATCHING_SOURCE_BASE + "/" + entry["path"], timeout=session_budget(30)) as response:
        data = response.read()
    assert len(data) == entry["bytes"] and hashlib.sha256(data).hexdigest() == entry["sha256"], entry["path"]
    target = batching_dir / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(data)

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(fetch_batching_file, batching_manifest["files"]))
(batching_dir / "MANIFEST.json").write_bytes(batching_manifest_raw)
print("Verified batching runner files:", len(batching_manifest["files"]))


Install GMP headers and a separate Python 3.12 environment for the original planner. Its dependency and arithmetic guards remain intact. The Colab notebook kernel can continue using its existing Python version.


In [ ]:
subprocess.run(["sudo", "apt-get", "update"], check=True, timeout=session_budget(180))
subprocess.run(["sudo", "apt-get", "install", "-y", "libgmp-dev", "g++", "python3.12", "python3.12-venv"], check=True, timeout=session_budget(180))
venv_dir = FIFTH42_DIR / "pipeline-venv"
subprocess.run(["/usr/bin/python3.12", "-m", "venv", str(venv_dir)], check=True, timeout=session_budget(60))
PIPELINE_PYTHON = str(venv_dir / "bin/python")
subprocess.run([PIPELINE_PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "--only-binary=:all:", "-r", str(pipeline_dir / "requirements.txt")], check=True, timeout=session_budget(240))
PIPELINE_ENV = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", PYTHONHASHSEED="0")
subprocess.run([PIPELINE_PYTHON, "build.py", "--gpu"], cwd=FIFTH42_DIR, env=PIPELINE_ENV, check=True, timeout=session_budget(180))


Run the exact independent GMP preflight, then five randomized repetitions of three policies: 1/4/8 windows, map chunk 8, eight preparation workers. Each trial makes 12 passes over the nine original windows. This is a longer repeated-input comparison, not production coverage or a guarantee of thermal steady state. If a run fails, run the final export cell to download its report/log.


In [ ]:
benchmark_seconds = int(min(480, session_budget(510) - 30))
if benchmark_seconds < 30:
    raise TimeoutError("Insufficient session budget for the bounded benchmark")
command = [PIPELINE_PYTHON, "batching_sustained/batch_bench.py", "--backend", "gpu", "--workers", "8", "--batch-windows", "1", "4", "8", "--map-chunksizes", "8", "--passes", "12", "--repeats", "5", "--max-seconds", str(benchmark_seconds)]
if UNITS_PER_HOUR is not None:
    command += ["--units-per-hour", str(float(UNITS_PER_HOUR))]
log_path = FIFTH42_DIR / "results/batching-run.log"
try:
    with log_path.open("w") as log:
        completed = subprocess.run(command, cwd=FIFTH42_DIR, env=PIPELINE_ENV, stdout=log, stderr=subprocess.STDOUT, timeout=session_budget(540))
finally:
    if log_path.exists():
        print(log_path.read_text())
completed.check_returncode()


In [ ]:
report = json.loads((FIFTH42_DIR / "results/batching-benchmark.json").read_text())
print(json.dumps({key: report.get(key) for key in ["status", "hardware", "workers", "replay_passes", "variants", "policy", "preflight", "summary", "elapsed_seconds", "limitations"]}, indent=2))

if report.get("status") == "passed":
    ratios = []
    for repeat in range(5):
        times = {row["preparation_batch_windows"]: row["pipeline_seconds"] for row in report["trials"] if row["repeat"] == repeat}
        ratios.append(times[4] / times[8])
    print("Eight-window throughput / four-window throughput, each matched repetition:", ratios)
    print("Above 1 favors eight; below 1 favors four. Assess variability and payload sizes before choosing.")


Download Fifth42_Sustained_Batching_Results.zip before disconnecting. It contains all 540 per-pass/per-region journals for a complete default run. Record compute-unit rates/balances separately if visible. A near tie between four and eight windows is a valid result.


In [ ]:
import zipfile
from google.colab import files
results_dir = FIFTH42_DIR / "results"
paths = [results_dir / name for name in ["batching-benchmark.json", "build.json", "batching-run.log"] if (results_dir / name).is_file()]
report_path = results_dir / "batching-benchmark.json"
if report_path.is_file():
    result_report = json.loads(report_path.read_text())
    journal_name = result_report.get("journal_directory", "")
    if journal_name:
        assert Path(journal_name).name == journal_name and journal_name.startswith("batching-journals-")
        paths += sorted(path for path in (results_dir / journal_name).rglob("*") if path.is_file() and path.suffix in [".json", ".jsonl"])
for name in ["SHA256SUMS", "data/export.json", "pipeline/MANIFEST.json", "pipeline/SOURCE_PROVENANCE.json", "pipeline/requirements.txt", "batching_sustained/MANIFEST.json"]:
    path = FIFTH42_DIR / name
    if path.is_file():
        paths.append(path)
assert sum(path.stat().st_size for path in paths) <= 96 * 1024**2, "Result export exceeds its finite size allowance"
result_zip = FIFTH42_DIR / "Fifth42_Sustained_Batching_Results.zip"
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    result_hashes = []
    for path in paths:
        rel = path.relative_to(FIFTH42_DIR)
        archive.write(path, str(rel))
        result_hashes.append(hashlib.sha256(path.read_bytes()).hexdigest() + "  " + str(rel))
    archive.writestr("RESULT_SHA256SUMS", "\n".join(result_hashes) + "\n")
print("Result ZIP SHA256:", hashlib.sha256(result_zip.read_bytes()).hexdigest())
files.download(str(result_zip))
